In [1]:
import pandas as pd 
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, roc_auc_score
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline

In [2]:
df = pd.read_csv('fraudTrain.csv')

In [3]:
df.sample(2)

,Unnamed: 0,trans_date_trans_time,cc_num,merchant,category,amt,first,last,gender,street,...,lat,long,city_pop,job,dob,trans_num,unix_time,merch_lat,merch_long,is_fraud
435185,435185,2019-07-13 21:39:36,676298633337,fraud_Schiller Ltd,personal_care,1.08,Jessica,Werner,F,4529 Cannon Alley,...,39.4125,-80.6352,255,Chief Executive Officer,1971-12-10,47f6ef9c1d72f4f110a9af8696282977,1342215576,38.550032,-81.250983,0
687385,687385,2019-10-20 12:32:51,4069975342931683,"fraud_Lang, Towne and Schuppe",kids_pets,44.15,Kimberly,Martin,F,1943 Dennis Inlet Suite 145,...,38.4257,-81.9943,21902,Sub,1980-09-18,fb0c1ed3eaa3d10fda89e49408d3f58a,1350736371,38.610456,-82.119064,0


In [4]:
df = df.drop(columns=['first','last','job','cc_num'])

In [5]:
# Check how many rows are duplicated at data set
print(f"Total duplicates: {df.duplicated().sum()}")

# View the duplication row
duplicate_rows = df[df.duplicated(keep=False)]
print(duplicate_rows.head())

Total duplicates: 0
Empty DataFrame
Columns: [Unnamed: 0, trans_date_trans_time, merchant, category, amt, gender, street, city, state, zip, lat, long, city_pop, dob, trans_num, unix_time, merch_lat, merch_long, is_fraud]
Index: []


In [6]:
# date and time extraction 
def date_time_extraction(df):
    df['trans_date_trans_time'] = pd.to_datetime(df['trans_date_trans_time'])

    df['hour'] = df['trans_date_trans_time'].dt.hour.astype(int)
    df['day'] = df['trans_date_trans_time'].dt.day.astype(int)
    df['month'] = df['trans_date_trans_time'].dt.month.astype(int)
    df['day_of_week'] = df['trans_date_trans_time'].dt.weekday.astype(int)
    return df

## Date of birth extraction
def dob(df):
    df['dob'] = pd.to_datetime(df['dob'])
    df['age'] = (df['trans_date_trans_time'] - df['dob']).dt.days // 365
    return df

## this method will calculate the distance between the merchant and the spender
## we are using this as distance could be a key factor in transaction frauds
def distance(df):
    from geopy.distance import geodesic  ## this is used to calculate the distance between two points on the earth
    df['distance'] = df.apply(
        lambda x: geodesic((x['lat'], x['long']),
                           (x['merch_lat'], x['merch_long'])).km,
        axis=1
    )
    return df

In [7]:
# to extract dates and years
df = date_time_extraction(df)

In [8]:
# to know the age of the spender
df = dob(df)

In [9]:
## this is to calculate distance b/w merchant and spender
df = distance(df)

In [10]:
df = df.drop(columns=['trans_date_trans_time', 'lat', 'long', 'merch_lat', 'merch_long',
                      'trans_num', 'street', 'zip', 'merchant', 'city', 'state',
                      'dob'])  # dropping dob since age is already extracted from it

In [11]:
df.sample(2)

,Unnamed: 0,category,amt,gender,city_pop,unix_time,is_fraud,hour,day,month,day_of_week,age,distance
232363,232363,entertainment,79.27,F,34882,1335538397,0,14,27,4,5,48,82.569466
1162410,1162410,kids_pets,19.06,M,206,1367260202,0,18,29,4,2,52,90.233853


In [12]:
df = pd.get_dummies(df, columns=['category'], drop_first=True)

In [13]:
bool_cols = df.select_dtypes(include='bool').columns
df[bool_cols] = df[bool_cols].astype(int)

In [14]:
df.columns

Index(['Unnamed: 0', 'amt', 'gender', 'city_pop', 'unix_time', 'is_fraud',
       'hour', 'day', 'month', 'day_of_week', 'age', 'distance',
       'category_food_dining', 'category_gas_transport',
       'category_grocery_net', 'category_grocery_pos',
       'category_health_fitness', 'category_home', 'category_kids_pets',
       'category_misc_net', 'category_misc_pos', 'category_personal_care',
       'category_shopping_net', 'category_shopping_pos', 'category_travel'],
      dtype='str')

## train test split

In [15]:
x = df.drop(columns=['is_fraud'])
y = df['is_fraud']

# encode gender before splitting
x['gender'] = x['gender'].map({'M': 0, 'F': 1})

x.sample(2)

,Unnamed: 0,amt,gender,city_pop,unix_time,hour,day,month,day_of_week,age,...,category_grocery_pos,category_health_fitness,category_home,category_kids_pets,category_misc_net,category_misc_pos,category_personal_care,category_shopping_net,category_shopping_pos,category_travel
471356,471356,84.02,1,3508,1343345129,23,26,7,4,73,...,0,0,1,0,0,0,0,0,0,0
655258,655258,78.24,1,7155,1349514198,9,6,10,6,42,...,0,0,0,0,0,0,0,0,0,0


In [16]:
x_tr, x_tst, y_tr, y_tst = train_test_split(x, y, test_size=0.2, random_state=44)

# check class distribution before balancing
print("Training set class distribution:")
print(y_tr.value_counts())
print("\nTest set class distribution:")
print(y_tst.value_counts())

Training set class distribution:
is_fraud
0    1031297
1       6043
Name: count, dtype: int64

Test set class distribution:
is_fraud
0    257872
1      1463
Name: count, dtype: int64


## column transformer

In [17]:
# gender is already encoded above so no need for OneHotEncoder anymore
# we use remainder='passthrough' to keep all other columns
trf = ColumnTransformer(transformers=[
    ('passthrough', 'passthrough', x_tr.columns.tolist())
], remainder='drop')

## pipeline with SMOTE + Random Forest
### WHY we changed from AdaBoost:
### 1. AdaBoost does not support class_weight='balanced'
### 2. Random Forest handles imbalanced data much better with class_weight
### 3. SMOTE synthetically creates minority class (fraud) samples to balance training data
### NOTE: We use imblearn's Pipeline (not sklearn's) so SMOTE only applies on training data

In [18]:
pipe = Pipeline(steps=[
    ('smote', SMOTE(random_state=42)),
    ('clf', RandomForestClassifier(
        class_weight='balanced',  # penalizes missing fraud cases more
        n_estimators=100,
        random_state=42,
        n_jobs=-1  # uses all CPU cores to speed up training
    ))
])

pipe

,steps,"[('smote', ...), ('clf', ...)]"
,transform_input,None
,memory,None
,verbose,False
,sampling_strategy,'auto'
,random_state,42
,k_neighbors,5
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2


In [19]:
pipe.fit(x_tr, y_tr)

,steps,"[('smote', ...), ('clf', ...)]"
,transform_input,None
,memory,None
,verbose,False
,sampling_strategy,'auto'
,random_state,42
,k_neighbors,5
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2


In [20]:
y_pred = pipe.predict(x_tst)

In [21]:
# NOTE: accuracy is a misleading metric for imbalanced data
# focus on recall and f1-score for class 1 (fraud) instead
print("Accuracy:", accuracy_score(y_tst, y_pred))

Accuracy: 0.9949563306148418


In [22]:
print("y_tst value counts:\n", y_tst.value_counts())
print("\ny_pred value counts:\n", pd.Series(y_pred).value_counts())

y_tst value counts:
 is_fraud
0    257872
1      1463
Name: count, dtype: int64

y_pred value counts:
 0    257180
1      2155
Name: count, dtype: int64


In [23]:
# full classification report - focus on class 1 precision, recall and f1
print(classification_report(y_tst, y_pred))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00    257872
           1       0.54      0.79      0.64      1463

    accuracy                           0.99    259335
   macro avg       0.77      0.89      0.82    259335
weighted avg       1.00      0.99      1.00    259335



In [24]:
# ROC-AUC score - best metric for imbalanced classification
y_proba = pipe.predict_proba(x_tst)[:, 1]
print("ROC-AUC Score:", roc_auc_score(y_tst, y_proba))

ROC-AUC Score: 0.9822160745706454


## Threshold Tuning
### By default predict() uses 0.5 as cutoff
### For fraud detection we prefer catching more frauds (high recall)
### so we lower the threshold - this increases recall but may lower precision

In [25]:
# try different thresholds and pick the one with best recall for fraud
for threshold in [0.5, 0.4, 0.3, 0.2]:
    y_pred_thresh = (y_proba >= threshold).astype(int)
    print(f"\n--- Threshold: {threshold} ---")
    print(classification_report(y_tst, y_pred_thresh))


--- Threshold: 0.5 ---
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    257872
           1       0.52      0.79      0.63      1463

    accuracy                           0.99    259335
   macro avg       0.76      0.89      0.81    259335
weighted avg       1.00      0.99      1.00    259335


--- Threshold: 0.4 ---
              precision    recall  f1-score   support

           0       1.00      0.99      1.00    257872
           1       0.42      0.82      0.55      1463

    accuracy                           0.99    259335
   macro avg       0.71      0.91      0.77    259335
weighted avg       1.00      0.99      0.99    259335


--- Threshold: 0.3 ---
              precision    recall  f1-score   support

           0       1.00      0.99      0.99    257872
           1       0.32      0.84      0.47      1463

    accuracy                           0.99    259335
   macro avg       0.66      0.92      0.73    259335
we

In [26]:
# use your chosen threshold here (e.g. 0.3 for higher fraud recall)
best_threshold = 0.3
y_pred_final = (y_proba >= best_threshold).astype(int)

print("Final Confusion Matrix:")
print(confusion_matrix(y_tst, y_pred_final))
print("\nFinal Classification Report:")
print(classification_report(y_tst, y_pred_final))

Final Confusion Matrix:
[[255289   2583]
 [   232   1231]]

Final Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.99      0.99    257872
           1       0.32      0.84      0.47      1463

    accuracy                           0.99    259335
   macro avg       0.66      0.92      0.73    259335
weighted avg       1.00      0.99      0.99    259335

